In [1]:
import xarray as xr
import numpy as np
import xarray as xr
import pandas as pd
import xesmf as xe

In [2]:
import os
import gc

In [3]:
# import h5py
# h5py._errors.silence_errors()


In [ ]:
from dask.distributed import Client, LocalCluster
cluster = LocalCluster()
client = Client(cluster)
client

In [ ]:
directory = ("/media/shrutee/Expansion1/ai_models/long_run/gc/")
data_files = [file for file in os.listdir(directory) if file.endswith("0000.nc")]

# Pangu

In [ ]:

for i,fil in enumerate(data_files[:]):

    data = xr.open_dataset(directory+fil, engine = "cfgrib", 
         backend_kwargs={"filter_by_keys": {"typeOfLevel": "isobaricInhPa"}}, chunks = {'step':16})

    latitudeN = 40
    latitudeS = -latitudeN
    dt = data[['u', 'v', 't', 'q', 'z']].sel(latitude = slice(latitudeN+1, latitudeS-1, ))
    del data; gc.collect()
    
    #regridding data
    arr = dt
    ds_out = xr.Dataset(
        {
            "latitude": (["latitude"],np.arange(latitudeN, latitudeS-1, -1), {"units": "degrees_north"}),
            "longitude": (["longitude"], np.arange(0, 360, 1), {"units": "degrees_east"}),
        }
    )
    regridder = xe.Regridder(dt, ds_out, "conservative")
    dt = regridder(arr, keep_attrs=True)
    del regridder, ds_out; gc.collect()
    dt = dt.compute()
    
    #changing step
    reference_time = pd.Timestamp(dt.time.values)
    dt['step'] = reference_time + dt.step
  
    uw = dt.u
    vw = dt.v
    tempw = dt.t
#     qw = dt.q #convert to fraction
    zw = dt.z
    qw = dt.q
#     rhw = dt.r/100 
    
#     epsilon = 0.622
#     t_celsius = tempw - 273.15 #changing to celsius
#     sat_vap_press = 611*np.exp(17.27*t_celsius/(237.3+t_celsius))
#     vap_press = rhw*sat_vap_press
#     pressure = dt.isobaricInhPa.broadcast_like(vap_press)*100
#     mixing_ratio = epsilon * vap_press/(pressure - vap_press)
#     q = mixing_ratio/(1+mixing_ratio)*1e3
    
#     del dt,t_celsius, sat_vap_press, vap_press, pressure, mixing_ratio, rhw; gc.collect()
#     uu = uw.resample(step = '1D').mean('step')
#     vv = vw.resample(step = '1D').mean('step')
#     temp = tempw.resample(step = '1D').mean('step')
#     z = zw.resample(step = '1D').mean('step')
#     q = qw.resample(step = '1D').mean('step')
    
    dx = 111e3 * np.cos(np.radians(uw.latitude))
    du_dx = uw.differentiate('longitude')/dx
    dv_dy = vw.differentiate('latitude')/111e3
    conv = du_dx+ dv_dy
    common_dims = ["step", "isobaricInhPa", "latitude", "longitude"]
    common_coords = dict(longitude=vw.longitude, isobaricInhPa=vw.isobaricInhPa,
                         latitude=vw.latitude,step=vw.step)
    
    w =  xr.DataArray(
    np.full( vw.shape, np.nan),
    dims=common_dims,
    coords=common_coords,
    name="vertical_velocity")
    
    for l,m in enumerate(vw.isobaricInhPa.values[:]):
        w[:,l,:,:] = -conv.sel(isobaricInhPa = slice(1000, m)).dropna("isobaricInhPa").integrate("isobaricInhPa")*100
    
    
   
 
    uw.name = "u_wind";vw.name = "v_wind";tempw.name = 'temp'; qw.name='q';zw.name='geop';
    w.name ="vertical_velocity"
    conv.name = "divergence"; #dw.name = 'w_wind'
    ds = xr.merge([uw,vw , tempw, qw, zw, conv, w])
    
    ds = ds.expand_dims({'run_number': [i]})
    ds = ds.rename({"time":'init_time', 'step':'time', 'isobaricInhPa':'level'})
    ds.to_netcdf(f"/media/shrutee/Expansion1/ai_models/long_run/lr/pangu_{i}.nc")
    
    del ds, uw,vw , tempw, qw, zw, conv, w, ; gc.collect()
    print("done")

# FC2

In [ ]:

for i,fil in enumerate(data_files[:]):

    data = xr.open_dataset(directory+fil, engine = "cfgrib", 
         backend_kwargs={"filter_by_keys": {"typeOfLevel": "isobaricInhPa"}}, chunks = {'step':16})

    latitudeN = 40
    latitudeS = -latitudeN
    dt = data[['u', 'v', 't', 'q', 'z']].sel(latitude = slice(latitudeN+1, latitudeS-1, ))
    del data; gc.collect()
    
    #regridding data
    arr = dt
    ds_out = xr.Dataset(
        {
            "latitude": (["latitude"],np.arange(latitudeN, latitudeS-1, -1), {"units": "degrees_north"}),
            "longitude": (["longitude"], np.arange(0, 360, 1), {"units": "degrees_east"}),
        }
    )
    regridder = xe.Regridder(dt, ds_out, "conservative")
    dt = regridder(arr, keep_attrs=True)
    del regridder, ds_out; gc.collect()
    dt = dt.compute()
    #changing step
    reference_time = pd.Timestamp(dt.time.values)
    dt['step'] = reference_time + dt.step
  
    uw = dt.u
    vw = dt.v
    tempw = dt.t

    zw = dt.z

    rhw = dt.r/100 
    
    epsilon = 0.622
    t_celsius = tempw - 273.15 #changing to celsius
    sat_vap_press = 611*np.exp(17.27*t_celsius/(237.3+t_celsius))
    vap_press = rhw*sat_vap_press
    pressure = dt.isobaricInhPa.broadcast_like(vap_press)*100
    mixing_ratio = epsilon * vap_press/(pressure - vap_press)
    q = mixing_ratio/(1+mixing_ratio)*1e3
    
    del dt,t_celsius, sat_vap_press, vap_press, pressure, mixing_ratio, rhw; gc.collect()

    dx = 111e3 * np.cos(np.radians(uw.latitude))
    du_dx = uw.differentiate('longitude')/dx
    dv_dy = vw.differentiate('latitude')/111e3
    conv = du_dx+ dv_dy
    common_dims = ["step", "isobaricInhPa", "latitude", "longitude"]
    common_coords = dict(longitude=vw.longitude, isobaricInhPa=vw.isobaricInhPa,
                         latitude=vw.latitude,step=vw.step)
    
    w =  xr.DataArray(
    np.full( vw.shape, np.nan), 
    dims=common_dims,
    coords=common_coords,
    name="vertical_velocity")
    
    for l,m in enumerate(vw.isobaricInhPa.values[:]):
        w[:,l,:,:] = -conv.sel(isobaricInhPa = slice(1000, m)).dropna("isobaricInhPa").integrate("isobaricInhPa")*100
    
    
   
    #saving final filtered dataset
    uw.name = "u_wind";vw.name = "v_wind";tempw.name = 'temp'; qw.name='q';zw.name='geop';
    w.name ="vertical_velocity"
    conv.name = "divergence"; #dw.name = 'w_wind'
    ds = xr.merge([uw,vw , tempw, qw, zw, conv, w])
    
    ds = ds.expand_dims({'run_number': [i]})
    ds = ds.rename({"time":'init_time', 'step':'time', 'isobaricInhPa':'level'})
    ds.to_netcdf(f"/media/shrutee/Expansion1/ai_models/long_run/lr/pangu_{i}.nc")
    
    del ds, uw,vw , tempw, qw, zw, conv, w, ; gc.collect()
    print("done")

# Graphcast

In [ ]:

for i,fil in enumerate(data_files[:]):

    data = xr.open_dataset(directory+fil, chunks = {'step':16})
    data = data.rename({'lat':'latitude', 'lon':'longitude', })

    latitudeN = 40
    latitudeS = -latitudeN
    dt = data[['u_component_of_wind', 'v_component_of_wind', 'temperature', 
               'specific_humidity', 'vertical_velocity', 'geopotential']].sel(latitude = slice(latitudeS, latitudeN),batch = 0)
    dt = dt.reindex(level=list(reversed(dt.level)))
    
    del data; gc.collect()
    
    #regridding data
    arr = dt
    ds_out = xr.Dataset(
        {
            "latitude": (["latitude"],np.arange(latitudeN, latitudeS-1, -1), {"units": "degrees_north"}),
            "longitude": (["longitude"], np.arange(0, 360, 1), {"units": "degrees_east"}),
        }
    )
    regridder = xe.Regridder(dt, ds_out, "conservative")
    dt = regridder(arr, keep_attrs=True)
    del regridder, ds_out; gc.collect()
    
    #changing step
    date_str = fil.split("_")[2]
    reference_time = pd.Timestamp(date_str)
    dt['time'] = reference_time + dt.time

    uw = dt.u_component_of_wind
    vw = dt.v_component_of_wind
    tempw = dt.temperature
    qw = dt.specific_humidity 
    ww = dt.vertical_velocity
    zw = dt.geopotential

    dx = 111e3 * np.cos(np.radians(uw.latitude))
    du_dx = uw.differentiate('longitude')/dx
    dv_dy = vw.differentiate('latitude')/111e3
    conv = (du_dx+ dv_dy).compute()
    del du_dx, dv_dy
    w =  xr.DataArray(
    np.full( vw.shape, np.nan),  # Fill with NaNs
    

    dims= ["time", "level", "latitude", "longitude"],
    coords= dict(longitude=vw.longitude, level=vw.level,
                         latitude=vw.latitude,step=vw.time),
    name="vertical_velocity")
    
    for l,m in enumerate(vw.level.values[:]):
        w[:,l,:,:] = -conv.sel(level = slice(1000, m)).dropna("level").integrate("level")*100
   
    uw.name = "u_wind";vw.name = "v_wind";tempw.name = 'temp'; qw.name='q';zw.name='geop';
    ww.name ="vertical_velocity_gc"; 
    conv.name = "divergence"; 
    ds = xr.merge([uw,vw , tempw, qw, zw, conv, ww, w])
    
    ds = ds.expand_dims({'run_number': [i]})
    ds.to_netcdf(f"/media/shrutee/Expansion1/ai_models/long_run/lr/gc_{i}.nc")
    
    del ds, uw,vw , tempw, qw, zw, conv, ww, w; gc.collect()
    print("done")